## Evaluation Baseline Models: 
Evaluate three-way baseline model provided by sustain-lc:
1. Rule-based controls. PID controls for CDU built into FMU if .env() is not called. CT always follows the wetbulb temp unless RL sets an offset on top of it. Main reason is to not violate thermodynamics, look more into the reasons why.
2. Multi-Agent PPO. Run multiagent_ca_ppo.py built on ca_ppo.py combined with the multiagent_ppo files.
3. Multihead PPO. Run multihead_ca_ppo.py built on ca_ppo.py combined with multihead_ppo files.

Run three control models in the same episode within the FMU, compare and plot results.

Afterwards, implment simple MLP control based on ca_ppo (or not), and compare to complete baselining results.

### 1. Rule-based PID Control (No agents/RL Strategy)

In [33]:
#import cell
import pyfmi
import numpy as np
import gymnasium as gym
import pandas as pd
import matplotlib.pyplot as plt
import copy as cp
from gymnasium import spaces
from scipy.special import softmax
from frontier_env import SmallFrontierModel

In [34]:
#Rule-Based (PID) Model Benchmark
sfm = SmallFrontierModel(exogen_gen_v = 2)

#vars needed for iteration, creating list of dicts, and rule-based action
episode = np.arange(0, 24*60*60+1, 15)
rule_rec = []
rule_based_action = {
            'cdu-cabinet-1': np.array([-0.2, -0.615, 1/3, 1/3, 1/3]),
            'cdu-cabinet-2': np.array([-0.2, -0.615, 1/3, 1/3, 1/3]),
            'cdu-cabinet-3': np.array([-0.2, -0.615, 1/3, 1/3, 1/3]),
            'cdu-cabinet-4': np.array([-0.2, -0.615, 1/3, 1/3, 1/3]),
            'cdu-cabinet-5': np.array([-0.2, -0.615, 1/3, 1/3, 1/3]),
            'cooling-tower-1': 4
        }

#reset FMU model env before running an ep
sfm.reset()

#creates a list of dictionaries with each dict being a single timestep entry, values are scaled
for k in episode:
    scaled_observation, reward, done, info = sfm.step(rule_based_action)
    temp_dict = {"timestep": k/15,
                 "reward": reward,
                 #Server-side Coolant Supply Temp
                 "block-1_boundary-1.T": scaled_observation['cdu-cabinet-1'][0],
                 "block-1_boundary-2.T": scaled_observation['cdu-cabinet-1'][1],
                 "block-1_boundary-3.T": scaled_observation['cdu-cabinet-1'][2],
                 "block-2_boundary-1.T": scaled_observation['cdu-cabinet-2'][0],
                 "block-2_boundary-2.T": scaled_observation['cdu-cabinet-2'][1],
                 "block-2_boundary-3.T": scaled_observation['cdu-cabinet-2'][2],
                 "block-3_boundary-1.T": scaled_observation['cdu-cabinet-3'][0],
                 "block-3_boundary-2.T": scaled_observation['cdu-cabinet-3'][1],
                 "block-3_boundary-3.T": scaled_observation['cdu-cabinet-3'][2],
                 "block-4_boundary-1.T": scaled_observation['cdu-cabinet-4'][0],
                 "block-4_boundary-2.T": scaled_observation['cdu-cabinet-4'][1],
                 "block-4_boundary-3.T": scaled_observation['cdu-cabinet-4'][2],
                 "block-5_boundary-1.T": scaled_observation['cdu-cabinet-5'][0],
                 "block-5_boundary-2.T": scaled_observation['cdu-cabinet-5'][1],
                 "block-5_boundary-3.T": scaled_observation['cdu-cabinet-5'][2],

                 #Cooling Tower Data
                 "CT-cell1.PFan": scaled_observation['cooling-tower-1'][0],
                 "CT-cell2.PFan": scaled_observation['cooling-tower-1'][1],
                 }
    
    temp_dict['total fan power'] = temp_dict['CT-cell1.PFan'] + temp_dict['CT-cell2.PFan']

    rule_rec.append(temp_dict)

FMU file loaded correctly: c:\Users\frank\NVAITC_files\NVAITC\sustain-lc/LC_Frontier_5Cabinet_4_17_25.fmu


In [ ]:
#Plotting
#conversion to unscaled physical values for plotting & evaluation
rule_df = pd.DataFrame(rule_rec)
unscaled_rule_df = rule_df.copy()

for i in np.arange(1, 5+1, 1):
    for k in np.arange(1, 3+1, 1):
        unscaled_rule_df[f'block-{i}_boundary-{k}.T'] = (rule_df[f'block-{i}_boundary-{k}.T']+1) * (100/2)

for col in ['CT-cell1.Pfan', 'CT-cell2.Pfan']:
    unscaled_rule_df[col] = (rule_df[col]+1) * 82027/2

unscaled_rule_df['total fan power'] = (rule_df['total fan power']+2) * 82028/2
   
#sum of total power drawn by 2 CT cell fans across all time steps
sum_power = unscaled_rule_df['total fan power'].sum()

plt.figure()
#testing out lambda, but will need to restructure list of dicts above later
plt.plot(unscaled_rule_df['timestep'], unscaled_rule_df['total fan power'])
plt.title("Rule-Based Control Total Fan Power Used at Each Timestep")
plt.xlabel("Timestep (15s/timestep)")
plt.ylabel("Total Fan Power")
plt.figtext(0.1, 0.01, f"Total Fan Power Drawn: {sum_power}")
plt.tight_layout()
plt.show()

KeyError: 'CT-cell1.Pfan'